### 1. Load and Preprocess Data

Prepare the dataset using tf.keras.utils.image_dataset_from_directory and apply data augmentation to increase diversity. Normalize pixel values to match the input requirements of the pre-trained model.

In [ ]:
import tensorflow as tf

# Load dataset
train_dataset = tf.keras.utils.image_dataset_from_directory(
   'path_to_train', image_size=(160, 160), batch_size=32, shuffle=True
)
validation_dataset = tf.keras.utils.image_dataset_from_directory(
   'path_to_validation', image_size=(160, 160), batch_size=32, shuffle=True
)
# Data augmentation
data_augmentation = tf.keras.Sequential([
   tf.keras.layers.RandomFlip('horizontal'),
   tf.keras.layers.RandomRotation(0.2),
])
# Normalize pixel values
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

### 2. Load Pre-Trained Model

Use a pre-trained model like MobileNetV2, excluding its top classification layers.

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
   input_shape=(160, 160, 3), include_top=False, weights='imagenet'
)
base_model.trainable = False # Freeze the base model

### 3. Add a New Classifier

Stack a global average pooling layer and a dense layer on top of the pre-trained model.

In [ ]:
global_average_layer = tf.keras.layers.GlobalAveragePooling2D()
prediction_layer = tf.keras.layers.Dense(1, activation='sigmoid')
inputs = tf.keras.Input(shape=(160, 160, 3))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = global_average_layer(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = prediction_layer(x)
model = tf.keras.Model(inputs, outputs)

### 4. Compile and Train

Compile the model with a low learning rate and train the classifier.

In [ ]:
model.compile(
   optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
   loss=tf.keras.losses.BinaryCrossentropy(),
   metrics=['accuracy']
)
history = model.fit(train_dataset, validation_data=validation_dataset, epochs=10)

### 5. Fine-Tune the Model

Unfreeze the top layers of the base model and fine-tune them with a lower learning rate.

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:100]: # Freeze lower layers
   layer.trainable = False
model.compile(
   optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.00001),
   loss=tf.keras.losses.BinaryCrossentropy(),
   metrics=['accuracy']
)
fine_tune_history = model.fit(train_dataset, validation_data=validation_dataset, epochs=10)

### 6. Evaluate and Predict

Evaluate the model on a test dataset and make predictions.

In [ ]:
loss, accuracy = model.evaluate(test_dataset)
print(f'Test accuracy: {accuracy}')
predictions = model.predict(test_dataset)

Summary

Transfer learning in TensorFlow involves reusing pre-trained models like MobileNetV2 for new tasks. Feature extraction is ideal for small datasets, while fine-tuning can further improve performance by adapting the model's higher-level features to the new dataset. This approach is efficient and reduces the need for extensive computational resources.